# Spotify Playlist Success Analysis

This notebook focuses on one question: **what playlist characteristics are associated with success in this sample?**

The notebook is intentionally explicit. It moves from raw data, to cleaning, to success definition, to correlation screening, to simple models that quantify how much variation the candidate features explain.


## 1. Setup

Import libraries, set notebook display options, and define helper functions for repeated analysis steps.


In [ ]:
import ast
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.anova import anova_lm
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

def make_hashable(value):
    """Convert lists to tuples so pandas nunique works on object columns like tokens."""
    if isinstance(value, list):
        return tuple(value)
    return value

def screen_spearman(data, target, features, sample_label):
    """Compute Spearman correlations and BH-adjusted p-values for a feature list."""
    rows = []
    for feature in features:
        sub = data[[feature, target]].dropna()
        if sub.empty or sub[feature].nunique() < 2 or sub[target].nunique() < 2:
            rows.append({
                'sample': sample_label,
                'feature': feature,
                'spearman_rho': np.nan,
                'p_value': np.nan,
                'n': len(sub)
            })
            continue
        rho, p = spearmanr(sub[feature], sub[target])
        rows.append({
            'sample': sample_label,
            'feature': feature,
            'spearman_rho': rho,
            'p_value': p,
            'n': len(sub)
        })

    out = pd.DataFrame(rows)
    out['p_adj_bh'] = np.nan
    valid = out['p_value'].notna()
    if valid.any():
        out.loc[valid, 'p_adj_bh'] = multipletests(out.loc[valid, 'p_value'], method='fdr_bh')[1]
    out['significant_bh_5pct'] = out['p_adj_bh'] < 0.05
    return out.sort_values('spearman_rho', key=lambda s: s.abs(), ascending=False)

def summarize_success_gap(data, success_col, features):
    """Compare medians and means for successful vs non-successful playlists."""
    rows = []
    success = data[data[success_col] == 1]
    rest = data[data[success_col] == 0]
    for feature in features:
        success_median = success[feature].median()
        rest_median = rest[feature].median()
        rows.append({
            'feature': feature,
            'success_median': success_median,
            'rest_median': rest_median,
            'median_ratio': success_median / rest_median if pd.notna(rest_median) and rest_median != 0 else np.nan,
            'success_mean': success[feature].mean(),
            'rest_mean': rest[feature].mean()
        })
    return pd.DataFrame(rows)

def category_lift_table(data, column, success_col='success_top_decile_within_segment', min_success_count=100, top_n=12):
    """Show categories overrepresented in the successful group."""
    success_counts = data.loc[data[success_col] == 1, column].value_counts()
    success_share = data.loc[data[success_col] == 1, column].value_counts(normalize=True)
    overall_share = data[column].value_counts(normalize=True)

    out = pd.DataFrame({
        'success_count': success_counts,
        'success_share': success_share,
        'overall_share': overall_share
    }).dropna()

    out = out[out['success_count'] >= min_success_count].copy()
    out['lift'] = out['success_share'] / out['overall_share']
    return out.sort_values(['lift', 'success_count'], ascending=[False, False]).head(top_n)

def top_category_group(series, top_n=10):
    """Keep only the top N categories and collapse the rest to Other."""
    top = series.value_counts().head(top_n).index
    return np.where(series.isin(top), series, 'Other')

def coefficient_table(model):
    """Collect coefficients, p-values, and BH-adjusted p-values from a fitted model."""
    out = pd.DataFrame({
        'term': model.params.index,
        'coef': model.params.values,
        'p_value': model.pvalues.values
    })
    out['p_adj_bh'] = multipletests(out['p_value'], method='fdr_bh')[1]
    return out.sort_values('p_adj_bh')


## 2. Load Raw Data

Read the assignment dataset and inspect its basic shape.


In [ ]:
df_raw = pd.read_csv('playlist_revision_v05.txt', sep='\t')

print(f'Rows: {df_raw.shape[0]:,}')
print(f'Columns: {df_raw.shape[1]}')
display(df_raw.head())


## 3. Clean Data

The cleaning step is intentionally minimal and easy to defend:
- drop clearly non-informative columns
- remove exact duplicate rows
- convert `'-'` placeholders in genre and mood columns to missing values
- parse `tokens` into real Python lists
- parse the playlist URI and create a Spotify-owned segment flag


In [ ]:
df_clean = df_raw.copy()

drop_cols = ['owner_country', 'mau_premium_ratio', 'mau_previous_month_premium_ratio']
dash_cols = ['genre_1', 'genre_2', 'genre_3', 'mood_1', 'mood_2', 'mood_3']

duplicate_rows = int(df_clean.duplicated().sum())
dash_placeholders = int((df_clean[dash_cols] == '-').sum().sum())

df_clean = df_clean.drop(columns=drop_cols).drop_duplicates().copy()
df_clean[dash_cols] = df_clean[dash_cols].replace('-', np.nan)
df_clean['tokens'] = df_clean['tokens'].apply(ast.literal_eval)

uri_parts = df_clean['playlist_uri'].str.split(':', expand=True)
df_clean['uri_owner'] = uri_parts[2]
df_clean['playlist_id'] = uri_parts[4]
df_clean['is_spotify_owned'] = df_clean['owner'].eq('spotify')
df_clean['owner_playlist_count'] = df_clean.groupby('owner')['playlist_uri'].transform('count')

print(f'Rows: {df_raw.shape[0]:,} -> {df_clean.shape[0]:,}')
print(f'Columns: {df_raw.shape[1]} -> {df_clean.shape[1]}')
print(f'Exact duplicate rows removed: {duplicate_rows:,}')
print(f'Dash placeholders converted to missing: {dash_placeholders:,}')
print(f'Spotify-owned playlists: {df_clean["is_spotify_owned"].sum():,}')
display(df_clean.head())


## 4. Define Success Metrics

At this stage, the notebook moves from broad EDA to a working definition of success.

### Why this definition?
- `monthly_stream30s` is more stable than daily `streams`.
- It already focuses on more meaningful listening than raw starts.
- The data show that owner self-listening can be large, especially outside Spotify-owned playlists.
- From a platform perspective, it is reasonable to prioritize listening **beyond the owner**.

### Primary success metric
- `non_owner_monthly_stream30s = max(monthly_stream30s - monthly_owner_stream30s, 0)`

### Supporting metrics
- `retention_rate = mau_both_months / mau_previous_month`
- `skip_rate = skippers / users`
- `stream30s_rate_today = stream30s / streams`
- `owner_stream_share = monthly_owner_stream30s / monthly_stream30s`

### Binary success label
To compare successful playlists with the rest, create a within-segment top-decile flag. This avoids defining success only as “being a Spotify-owned playlist.”


In [ ]:
df_clean['non_owner_monthly_stream30s'] = (
    df_clean['monthly_stream30s'] - df_clean['monthly_owner_stream30s']
).clip(lower=0)
df_clean['retention_rate'] = df_clean['mau_both_months'] / df_clean['mau_previous_month'].replace(0, np.nan)
df_clean['skip_rate'] = df_clean['skippers'] / df_clean['users'].replace(0, np.nan)
df_clean['stream30s_rate_today'] = df_clean['stream30s'] / df_clean['streams'].replace(0, np.nan)
df_clean['owner_stream_share'] = df_clean['monthly_owner_stream30s'] / df_clean['monthly_stream30s'].replace(0, np.nan)
df_clean['log_success'] = np.log1p(df_clean['non_owner_monthly_stream30s'])
df_clean['one_artist'] = (df_clean['n_artists'] == 1).astype(int)
df_clean['segment'] = np.where(df_clean['is_spotify_owned'], 'spotify_owned', 'non_spotify')

df_clean['success_top_decile_within_segment'] = (
    df_clean.groupby('segment')['non_owner_monthly_stream30s']
    .transform(lambda s: s >= s.quantile(0.9))
    .astype(int)
)

segment_summary = df_clean.groupby('segment').agg(
    playlists=('playlist_uri', 'count'),
    median_non_owner_monthly_stream30s=('non_owner_monthly_stream30s', 'median'),
    median_monthly_stream30s=('monthly_stream30s', 'median'),
    median_retention_rate=('retention_rate', 'median'),
    median_owner_stream_share=('owner_stream_share', 'median')
)

success_label_summary = df_clean.groupby('segment')['success_top_decile_within_segment'].agg(['sum', 'mean'])
success_label_summary.columns = ['successful_playlists', 'success_share']

display(segment_summary)
display(success_label_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df_clean, x='log_success', hue='segment', common_norm=False, bins=50, element='step', ax=axes[0])
axes[0].set_title('Distribution of log1p(non-owner monthly streams)')
axes[0].set_xlabel('log1p(non_owner_monthly_stream30s)')

sns.boxplot(data=df_clean, x='segment', y='log_success', ax=axes[1])
axes[1].set_title('Success distribution by segment')
axes[1].set_xlabel('segment')
axes[1].set_ylabel('log_success')

plt.tight_layout()


## 5. Correlation Screening

Use Spearman correlations because the variables are skewed and many relationships are nonlinear.

Important note: with a dataset this large, many p-values will be extremely small. The p-values matter, but **effect size** (`rho`) and **explained variation** (`R²`) matter more.

I separate the screening into two feature families:
- **Structural / context features**: playlist size, diversity, and owner history
- **Behavioral signatures**: metrics that may move closely with success, but are closer to the outcome itself


In [ ]:
samples = {
    'overall': df_clean,
    'non_spotify': df_clean[df_clean['segment'] == 'non_spotify'],
    'spotify_owned': df_clean[df_clean['segment'] == 'spotify_owned']
}

structural_features = ['n_tracks', 'n_artists', 'n_albums', 'n_local_tracks', 'owner_playlist_count']
behavioral_features = ['retention_rate', 'skip_rate', 'stream30s_rate_today', 'owner_stream_share']

structural_corr_results = {}
for sample_name, sample_frame in samples.items():
    structural_corr_results[sample_name] = screen_spearman(sample_frame, 'log_success', structural_features, sample_name)
    print(sample_name)
    display(structural_corr_results[sample_name])


In [ ]:
behavioral_corr_results = {}
for sample_name, sample_frame in samples.items():
    behavioral_corr_results[sample_name] = screen_spearman(sample_frame, 'log_success', behavioral_features, sample_name)
    print(sample_name)
    display(behavioral_corr_results[sample_name])


## 6. Compare Successful Playlists With the Rest

Use the within-segment top-decile success label to make the differences more concrete.

The main comparison focuses on the **non-Spotify** segment because it represents nearly the entire sample and avoids collapsing the analysis into a trivial Spotify-vs-not-Spotify comparison.


In [ ]:
non_spotify = df_clean[df_clean['segment'] == 'non_spotify'].copy()

one_artist_summary = non_spotify.groupby('one_artist').agg(
    playlists=('playlist_uri', 'count'),
    median_non_owner_monthly_stream30s=('non_owner_monthly_stream30s', 'median'),
    median_log_success=('log_success', 'median'),
    median_owner_stream_share=('owner_stream_share', 'median')
)

success_gap_features = [
    'n_tracks',
    'n_artists',
    'n_albums',
    'n_local_tracks',
    'owner_playlist_count',
    'one_artist',
    'retention_rate',
    'skip_rate',
    'stream30s_rate_today',
    'owner_stream_share'
]

non_spotify_success_gap = summarize_success_gap(
    non_spotify,
    success_col='success_top_decile_within_segment',
    features=success_gap_features
)

genre_lift_non_spotify = category_lift_table(non_spotify, 'genre_1')
mood_lift_non_spotify = category_lift_table(non_spotify, 'mood_1')

display(one_artist_summary)
display(non_spotify_success_gap)
display(genre_lift_non_spotify)
display(mood_lift_non_spotify)


## 7. Explanatory Models

Use simple OLS models with `log_success` as the outcome.

These models are **not causal**. They quantify conditional associations and how much variation the observed features explain.

I estimate three models for the non-Spotify sample:
- **Structural only**: playlist size, diversity, and owner history
- **Structural + content**: add grouped genre and mood indicators
- **Behavioral signatures**: engagement-adjacent variables such as owner share and skip rate

I also fit a small Spotify-owned structural model as an exploratory check, but I treat that segment more cautiously because it contains only a few hundred playlists.


In [ ]:
model_df_non = non_spotify.copy()
model_df_spotify = df_clean[df_clean['segment'] == 'spotify_owned'].copy()

for col in ['genre_1', 'mood_1']:
    model_df_non[f'{col}_grouped'] = top_category_group(model_df_non[col], top_n=10)

for frame in [model_df_non, model_df_spotify]:
    for col in ['n_tracks', 'n_artists', 'n_albums', 'n_local_tracks', 'owner_playlist_count']:
        frame[f'log_{col}'] = np.log1p(frame[col])

vif_input = model_df_non[
    ['log_n_tracks', 'log_n_artists', 'log_n_albums', 'log_n_local_tracks', 'log_owner_playlist_count']
].dropna().copy()
vif_input.insert(0, 'const', 1.0)

vif_table = []
for idx, column in enumerate(vif_input.columns):
    if column == 'const':
        continue
    vif_table.append({
        'feature': column,
        'vif': variance_inflation_factor(vif_input.values, idx)
    })
vif_table = pd.DataFrame(vif_table)

display(vif_table)


In [ ]:
structural_formula = (
    'log_success ~ log_n_tracks + log_n_artists + log_n_albums '
    '+ log_n_local_tracks + log_owner_playlist_count + one_artist'
)

content_formula = structural_formula + ' + C(genre_1_grouped) + C(mood_1_grouped)'
behavior_formula = 'log_success ~ retention_rate + skip_rate + stream30s_rate_today + owner_stream_share'
spotify_structural_formula = 'log_success ~ log_n_tracks + log_n_artists + log_n_albums + log_n_local_tracks + one_artist'

model_structural = smf.ols(structural_formula, data=model_df_non).fit()
model_content = smf.ols(content_formula, data=model_df_non).fit()
model_behavior = smf.ols(
    behavior_formula,
    data=model_df_non.dropna(subset=['retention_rate', 'skip_rate', 'stream30s_rate_today', 'owner_stream_share'])
).fit()
model_spotify_structural = smf.ols(spotify_structural_formula, data=model_df_spotify).fit()

model_fit_summary = pd.DataFrame([
    {
        'model': 'non_spotify_structural_only',
        'nobs': model_structural.nobs,
        'r2': model_structural.rsquared,
        'adj_r2': model_structural.rsquared_adj,
        'model_pvalue': model_structural.f_pvalue
    },
    {
        'model': 'non_spotify_structural_plus_content',
        'nobs': model_content.nobs,
        'r2': model_content.rsquared,
        'adj_r2': model_content.rsquared_adj,
        'model_pvalue': model_content.f_pvalue
    },
    {
        'model': 'non_spotify_behavioral_signatures',
        'nobs': model_behavior.nobs,
        'r2': model_behavior.rsquared,
        'adj_r2': model_behavior.rsquared_adj,
        'model_pvalue': model_behavior.f_pvalue
    },
    {
        'model': 'spotify_structural_exploratory',
        'nobs': model_spotify_structural.nobs,
        'r2': model_spotify_structural.rsquared,
        'adj_r2': model_spotify_structural.rsquared_adj,
        'model_pvalue': model_spotify_structural.f_pvalue
    }
])

content_anova = anova_lm(model_structural, model_content)

display(model_fit_summary)
display(content_anova)

plt.figure(figsize=(10, 4))
sns.barplot(data=model_fit_summary, x='model', y='adj_r2', color='#4C78A8')
plt.xticks(rotation=25, ha='right')
plt.title('Adjusted R-squared by model')
plt.ylabel('Adjusted R-squared')
plt.xlabel('')
plt.tight_layout()


In [ ]:
structural_coef_table = coefficient_table(model_structural)
content_coef_table = coefficient_table(model_content)
behavior_coef_table = coefficient_table(model_behavior)
spotify_coef_table = coefficient_table(model_spotify_structural)

display(structural_coef_table)
display(content_coef_table.head(25))
display(behavior_coef_table)
display(spotify_coef_table)


## 8. Review the Results

This final section turns the tables above into a concise review.

The goal is not to claim causality. The goal is to identify which features are **consistently associated** with the working success metric, and how much variation those features explain.


In [ ]:
top_structural_non_spotify = structural_corr_results['non_spotify'].iloc[0]
top_behavioral_non_spotify = behavioral_corr_results['non_spotify'].iloc[0]

print('Success definition')
print('- Primary metric: non_owner_monthly_stream30s')
print('- Binary label: top 10% within segment')
print()

print('Segment difference')
print(
    f"- Non-Spotify median non-owner monthly streams: {segment_summary.loc['non_spotify', 'median_non_owner_monthly_stream30s']:.1f}"
)
print(
    f"- Spotify-owned median non-owner monthly streams: {segment_summary.loc['spotify_owned', 'median_non_owner_monthly_stream30s']:.1f}"
)
print(
    f"- Non-Spotify median owner stream share: {segment_summary.loc['non_spotify', 'median_owner_stream_share']:.3f}"
)
print(
    f"- Spotify-owned median owner stream share: {segment_summary.loc['spotify_owned', 'median_owner_stream_share']:.3f}"
)
print()

print('Strongest non-Spotify structural correlation')
print(
    f"- {top_structural_non_spotify['feature']} with Spearman rho = {top_structural_non_spotify['spearman_rho']:.3f}"
)
print()

print('Strongest non-Spotify behavioral signature')
print(
    f"- {top_behavioral_non_spotify['feature']} with Spearman rho = {top_behavioral_non_spotify['spearman_rho']:.3f}"
)
print()

print('One-artist playlists in the non-Spotify sample')
print(
    f"- Median non-owner monthly streams, one-artist = {one_artist_summary.loc[1, 'median_non_owner_monthly_stream30s']:.1f}"
)
print(
    f"- Median non-owner monthly streams, multi-artist = {one_artist_summary.loc[0, 'median_non_owner_monthly_stream30s']:.1f}"
)
print()

print('Model fit')
print(
    f"- Structural features explain about {model_structural.rsquared_adj:.3f} adjusted R-squared in the non-Spotify sample."
)
print(
    f"- Adding grouped genre and mood features raises adjusted R-squared to about {model_content.rsquared_adj:.3f}."
)
print(
    f"- Behavioral signatures explain much more variation ({model_behavior.rsquared_adj:.3f}), but they are closer to the outcome itself and should be interpreted differently."
)
print(
    f"- The exploratory Spotify-only structural model explains about {model_spotify_structural.rsquared_adj:.3f} adjusted R-squared."
)
print()

print('Working interpretation')
print('- Playlist structure matters: larger and more diverse playlists tend to perform better on the primary success metric.')
print('- Success is strongly segmented: Spotify-owned playlists and non-Spotify playlists behave very differently.')
print('- One-artist playlists underperform on the non-owner success metric in the non-Spotify sample.')
print('- Content categories add statistically significant but modest incremental explanatory power on top of structure.')
print('- Because the sample is very large, significance is easy to obtain; effect size and explained variation are more informative than p-values alone.')
